In [ ]:
from pathlib import Path
import sys

In [ ]:
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))

In [ ]:
from IPython.core.magic import register_cell_magic

from gold.ddl import run_ddl


@register_cell_magic
def sql(line, cell):
    """%%sql cell: runs each statement. PRIMARY KEY, FOREIGN KEY and UNIQUE are only applied on Databricks."""
    skipped = run_ddl(spark, cell, gold=GOLD)
    if skipped:
        print(f"Skipped {len(skipped)} informational constraints (Databricks only):")
        for statement in skipped:
            print("  ", statement)


In [ ]:
from pipeline.catalog import ensure_schema_exists, get_schema_fqn
from pipeline.session import get_spark_session

spark = get_spark_session()

CATALOG = 'forge'
GOLD_SCHEMA = 'gold'
GOLD = get_schema_fqn(CATALOG, GOLD_SCHEMA)   # 'gold' locally, 'forge.gold' on Databricks
ensure_schema_exists(spark, CATALOG, GOLD_SCHEMA)


In [ ]:
%%sql

-- ============================================================
-- DDL: bridge_product_category
-- Gold Layer — Product-Category Many-to-Many Bridge
-- ============================================================
-- Grain:            One row per product–category assignment
-- Business Key:     product_category_id
-- Surrogate Key:    bridge_sk (deterministic xxhash64)
-- Source:           silver_product_category
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.bridge_product_category
(
    -- Surrogate key
    bridge_sk BIGINT NOT NULL,

    -- Business key
    product_category_id STRING NOT NULL,

    -- Foreign keys to dimensions
    product_sk BIGINT NOT NULL,
    category_sk BIGINT NOT NULL,

    -- Bridge attributes
    is_primary BOOLEAN,
    active_from DATE,
    active_to DATE,

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints
    CONSTRAINT bridge_product_category_pk PRIMARY KEY (bridge_sk),
    CONSTRAINT fk_bridge_product FOREIGN KEY (product_sk) REFERENCES ${gold}.dim_product(product_sk),
    CONSTRAINT fk_bridge_category FOREIGN KEY (category_sk) REFERENCES ${gold}.dim_category(category_sk)
)
USING DELTA
CLUSTER BY (product_sk, category_sk);

ALTER TABLE ${gold}.bridge_product_category ADD CONSTRAINT bridge_product_category_bk UNIQUE (product_category_id);


In [ ]:
%%sql

-- ============================================================
-- DDL: dim_address
-- Gold Layer — SCD Type 2 Address Dimension
-- ============================================================
-- Grain:           One row per address version
-- Business Key:    address_id
-- Surrogate Key:   address_sk (deterministic xxhash64)
-- Change Detection: attribute_hash (SHA-256 of tracked columns)
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.dim_address
(
    -- Surrogate key
    -- Deterministic hash of (address_id, effective_start_date)
    address_sk BIGINT NOT NULL,

    -- Business key
    address_id STRING NOT NULL,

    -- Foreign key to dim_customer
    customer_id STRING NOT NULL,

    -- Address attributes
    address_type STRING,
    address_line_1 STRING,
    address_line_2 STRING,
    city STRING,
    state_region STRING,
    country_subdivision_code STRING,
    postal_code STRING,
    country_code STRING,
    is_primary BOOLEAN,

    -- SCD Type 2 change detection
    attribute_hash STRING NOT NULL,

    -- SCD Type 2 versioning metadata
    effective_start_date TIMESTAMP NOT NULL,
    effective_end_date TIMESTAMP NOT NULL,
    is_current BOOLEAN NOT NULL,
    row_version INT NOT NULL,

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints
    CONSTRAINT pk_dim_address PRIMARY KEY (address_sk)

)
USING DELTA

CLUSTER BY (
    address_id,
    effective_start_date,
    effective_end_date
);

MERGE INTO 
        ${gold}.dim_address
    AS tgt_tbl
    USING
    (
        SELECT
            CAST(-1 AS BIGINT) AS address_sk,
            '-1' AS address_id,
            '-1' AS customer_id,
            'UNKNOWN' AS address_type,
            'UNKNOWN' AS address_line_1,
            NULL AS address_line_2,
            'UNKNOWN' AS city,
            NULL AS state_region,
            NULL AS country_subdivision_code,
            NULL AS postal_code,
            NULL AS country_code,
            FALSE AS is_primary,
            'SYSTEM' AS attribute_hash,
            TIMESTAMP '1900-01-01 00:00:00' AS effective_start_date,
            TIMESTAMP '9999-12-31 00:00:00' AS effective_end_date,
            TRUE AS is_current,
            0 AS row_version,
            'SYSTEM' AS _source_batch_id,
        'SYSTEM' AS _pipeline_run_id,
            current_timestamp() AS _gold_loaded_timestamp
    ) AS src_tbl

    ON tgt_tbl.address_sk = src_tbl.address_sk
    WHEN NOT MATCHED
    THEN INSERT *;

ALTER TABLE ${gold}.dim_address ADD CONSTRAINT ck_dim_address_row_version CHECK (row_version >= 0);
ALTER TABLE ${gold}.dim_address ADD CONSTRAINT ck_dim_address_effective_range CHECK (effective_end_date >= effective_start_date)

In [ ]:
%%sql

-- ============================================================
-- DDL: dim_category
-- Gold Layer — SCD Type 1 Category Dimension
-- ============================================================
-- Grain:            One row per category (current state only)
-- Business Key:     category_id
-- Surrogate Key:    category_sk (deterministic xxhash64)
-- Change Detection: attribute_hash (SHA-256 of tracked columns)
-- SCD Strategy:     Type 1 — overwrites in place, no history
-- ============================================================

-- ============================================================
-- Step 1: Create the dim_category table
-- ============================================================
-- Defines the Gold-layer dimension with surrogate key, business
-- key, descriptive attributes, a change-detection hash, and
-- audit columns.  PK and UNIQUE constraints are informational
-- (not enforced) but used by the query optimiser.
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.dim_category
(
    -- Surrogate key
    category_sk BIGINT NOT NULL,

    -- Business key
    category_id STRING NOT NULL,

    -- Category attributes
    category_name STRING NOT NULL,
    parent_category_id STRING,
    category_code STRING,
    is_active BOOLEAN,

    -- SCD Type 1 change-detection hash
    attribute_hash STRING NOT NULL,

    -- Audit metadata
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints (informational — used by the optimiser)
    CONSTRAINT dim_category_pk PRIMARY KEY (category_sk)
)
USING DELTA;


-- ============================================================
-- Step 2: Seed the unknown / default member row
-- ============================================================
-- Inserts a single "unknown" row (category_sk = -1) used by
-- fact tables when a category lookup fails.  The MERGE ensures
-- idempotency — the row is only inserted if it does not already
-- exist.
-- ============================================================

MERGE INTO ${gold}.dim_category AS tgt_tbl
USING
(
    SELECT
        CAST(-1 AS BIGINT)  AS category_sk,
        '-1'                AS category_id,
        'UNKNOWN'           AS category_name,
        NULL                AS parent_category_id,
        'UNKNOWN'           AS category_code,
        FALSE               AS is_active,
        'SYSTEM'            AS attribute_hash,
        'SYSTEM' AS _source_batch_id,
        'SYSTEM' AS _pipeline_run_id,
        current_timestamp() AS _gold_loaded_timestamp
) AS src_tbl
ON tgt_tbl.category_sk = src_tbl.category_sk
WHEN NOT MATCHED
THEN INSERT *;

ALTER TABLE ${gold}.dim_category ADD CONSTRAINT dim_category_bk UNIQUE (category_id);

In [ ]:
%%sql

-- ============================================================
-- DDL: dim_customer
-- Gold Layer — SCD Type 2 Customer Dimension
-- ============================================================
-- Grain:            One row per customer version
-- Business Key:     customer_id
-- Surrogate Key:    customer_sk (deterministic xxhash64)
-- Change Detection: attribute_hash (SHA-256 of tracked columns)
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.dim_customer
(
    -- Surrogate key
    -- Deterministic hash of (customer_id, effective_start_date)
    customer_sk BIGINT NOT NULL,

    -- Business key
    customer_id STRING NOT NULL,

    -- Customer attributes
    first_name STRING,
    last_name STRING,
    email_hash STRING,  -- salted SHA-256 of the normalised email; the raw address is never stored in Gold
    email_domain STRING,
    phone STRING,
    date_of_birth DATE,
    loyalty_status STRING,
    marketing_opt_in BOOLEAN,

    -- SCD Type 2 change detection
    attribute_hash STRING NOT NULL,

    -- SCD Type 2 versioning metadata
    effective_start_date TIMESTAMP NOT NULL,
    effective_end_date TIMESTAMP NOT NULL,
    is_current BOOLEAN NOT NULL,
    row_version INT NOT NULL,

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints
    CONSTRAINT pk_dim_customer PRIMARY KEY (customer_sk)
)
USING DELTA
CLUSTER BY (
        customer_id,
        effective_start_date
    );

MERGE INTO
        ${gold}.dim_customer
        AS tgt_tbl
    USING
    (
        SELECT
            CAST(-1 AS BIGINT) AS customer_sk,
            '-1' AS customer_id,
            'UNKNOWN' AS first_name,
            'UNKNOWN' AS last_name,
            NULL AS email_hash,
            NULL AS email_domain,
            NULL AS phone,
            NULL AS date_of_birth,
            'UNKNOWN' AS loyalty_status,
            FALSE AS marketing_opt_in,
            'SYSTEM' AS attribute_hash,
            TIMESTAMP '1900-01-01 00:00:00' AS effective_start_date,
            TIMESTAMP '9999-12-31 00:00:00' AS effective_end_date,
            TRUE AS is_current,
            0 AS row_version,
            'SYSTEM' AS _source_batch_id,
        'SYSTEM' AS _pipeline_run_id,
            current_timestamp() AS _gold_loaded_timestamp
    ) AS src_tbl

    ON tgt_tbl.customer_sk = src_tbl.customer_sk
    WHEN NOT MATCHED
    THEN INSERT *;

ALTER TABLE ${gold}.dim_customer ADD CONSTRAINT ck_dim_customer_effective_dates CHECK (effective_start_date <= effective_end_date);
ALTER TABLE ${gold}.dim_customer ADD CONSTRAINT ck_dim_customer_row_version CHECK (row_version >= 0)


In [ ]:
%%sql

-- ============================================================
-- DDL: dim_date
-- Gold Layer — SCD Type 0 Date Dimension
-- ============================================================
-- Grain:            One row per calendar date
-- Business Key:     date_key (YYYYMMDD integer)
-- Surrogate Key:    date_sk (deterministic xxhash64)
-- SCD Strategy:     Type 0 — immutable, loaded once
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.dim_date
(
    -- Surrogate key
    date_sk BIGINT NOT NULL,

    -- Business key (YYYYMMDD integer format)
    date_key INT NOT NULL,

    -- Calendar attributes
    full_date DATE,
    year INT,
    quarter INT,
    quarter_name STRING,
    month INT,
    month_name STRING,
    month_short_name STRING,
    week_of_year INT,
    iso_day_of_week INT,
    day_of_month INT,
    day_of_year INT,
    day_name STRING,
    year_month STRING,
    year_quarter STRING,
    calendar_month_start_date DATE,
    calendar_quarter_start_date DATE,
    calendar_year_start_date DATE,
    is_weekend BOOLEAN,
    is_month_start BOOLEAN,
    is_month_end BOOLEAN,
    is_quarter_start BOOLEAN,
    is_quarter_end BOOLEAN,
    is_year_start BOOLEAN,
    is_year_end BOOLEAN,
    _batch_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints (informational — used by the optimiser)
    CONSTRAINT dim_date_pk PRIMARY KEY (date_sk)
)
USING DELTA;

INSERT INTO ${gold}.dim_date
SELECT
    xxhash64(date_key) AS date_sk,
    date_key,
    full_date,
    year,
    quarter,
    quarter_name,
    month,
    month_name,
    month_short_name,
    week_of_year,
    iso_day_of_week,
    day_of_month,
    day_of_year,
    day_name,
    year_month,
    year_quarter,
    calendar_month_start_date,
    calendar_quarter_start_date,
    calendar_year_start_date,
    is_weekend,
    is_month_start,
    is_month_end,
    is_quarter_start,
    is_quarter_end,
    is_year_start,
    is_year_end,
    'SYSTEM' AS _batch_id,
    current_timestamp() AS _gold_loaded_timestamp
FROM (
    SELECT

        CAST(
            YEAR(full_date) * 10000
            + MONTH(full_date) * 100
            + DAY(full_date) AS INT
        ) AS date_key,

        full_date,
        YEAR(full_date) AS year,
        QUARTER(full_date) AS quarter,
        CONCAT('Q', QUARTER(full_date)) AS quarter_name,
        MONTH(full_date) AS month,
        DATE_FORMAT(full_date, 'MMMM') AS month_name,
        DATE_FORMAT(full_date, 'MMM') AS month_short_name,
        WEEKOFYEAR(full_date) AS week_of_year,
        WEEKDAY(full_date) + 1  AS iso_day_of_week,
        DAY(full_date) AS day_of_month,
        DAYOFYEAR(full_date) AS day_of_year,
        DATE_FORMAT(full_date, 'EEEE') AS day_name,
        DATE_FORMAT(full_date, 'yyyy-MM') AS year_month,
        CONCAT(YEAR(full_date), '-Q', QUARTER(full_date)) AS year_quarter,
        TRUNC(full_date, 'MM') AS calendar_month_start_date,
        CAST(DATE_TRUNC('QUARTER', full_date) AS DATE) AS calendar_quarter_start_date,
        TRUNC(full_date, 'YEAR') AS calendar_year_start_date,
        WEEKDAY(full_date) >= 5 AS is_weekend,
        DAY(full_date) = 1 AS is_month_start,
        full_date = LAST_DAY(full_date) AS is_month_end,
        MONTH(full_date) IN (1, 4, 7, 10) AND DAY(full_date) = 1 AS is_quarter_start,
        MONTH(full_date) IN (3, 6, 9, 12) AND full_date = LAST_DAY(full_date) AS is_quarter_end,
        MONTH(full_date) = 1  AND DAY(full_date) = 1 AS is_year_start,
        MONTH(full_date) = 12 AND DAY(full_date) = 31 AS is_year_end

    FROM (
        SELECT EXPLODE(SEQUENCE(DATE'2024-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS full_date
    ) date_spine
) date_attrs;


MERGE INTO ${gold}.dim_date AS date_tgt
USING
(
    SELECT
        CAST(-1 AS BIGINT)      AS date_sk,
        CAST(-1 AS INT)         AS date_key,
        CAST(NULL AS DATE)      AS full_date,
        CAST(NULL AS INT)       AS year,
        CAST(NULL AS INT)       AS quarter,
        'UNKNOWN'               AS quarter_name,
        CAST(NULL AS INT)       AS month,
        'UNKNOWN'               AS month_name,
        'UNK'                   AS month_short_name,
        CAST(NULL AS INT)       AS week_of_year,
        CAST(NULL AS INT)       AS iso_day_of_week,
        CAST(NULL AS INT)       AS day_of_month,
        CAST(NULL AS INT)       AS day_of_year,
        'UNKNOWN'               AS day_name,
        NULL                    AS year_month,
        NULL                    AS year_quarter,
        CAST(NULL AS DATE)      AS calendar_month_start_date,
        CAST(NULL AS DATE)      AS calendar_quarter_start_date,
        CAST(NULL AS DATE)      AS calendar_year_start_date,
        FALSE                   AS is_weekend,
        FALSE                   AS is_month_start,
        FALSE                   AS is_month_end,
        FALSE                   AS is_quarter_start,
        FALSE                   AS is_quarter_end,
        FALSE                   AS is_year_start,
        FALSE                   AS is_year_end,
        'SYSTEM' AS _batch_id,
        current_timestamp()     AS _gold_loaded_timestamp
) AS date_src
ON date_tgt.date_sk = date_src.date_sk
WHEN NOT MATCHED
THEN INSERT *;

ALTER TABLE ${gold}.dim_date ADD CONSTRAINT dim_date_bk UNIQUE (date_key);


In [ ]:
%%sql

-- ============================================================
-- DDL: dim_product
-- Gold Layer — SCD Type 2 Product Dimension
-- ============================================================
-- Grain:            One row per product version
-- Business Key:     product_id
-- Surrogate Key:    product_sk (deterministic xxhash64)
-- Change Detection: attribute_hash (SHA-256 of tracked columns)
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.dim_product
(
    -- Surrogate key
    -- Deterministic hash of (product_id, effective_start_date)
    product_sk BIGINT NOT NULL,

    -- Business key
    product_id STRING NOT NULL,

    -- Product attributes
    sku STRING,
    product_name STRING,
    brand STRING,
    unit_price DECIMAL(18,4),
    currency_code STRING,
    is_active BOOLEAN,

    -- SCD Type 2 change detection
    attribute_hash STRING NOT NULL,

    -- SCD Type 2 versioning metadata
    effective_start_date TIMESTAMP NOT NULL,
    effective_end_date TIMESTAMP NOT NULL,
    is_current BOOLEAN NOT NULL,
    row_version INT NOT NULL,

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- constraints
    CONSTRAINT pk_product_sk PRIMARY KEY (product_sk)
    
)
USING DELTA

 CLUSTER BY (
        product_id,
        effective_start_date,
        effective_end_date
);

MERGE INTO
        ${gold}.dim_product
    AS tgt_tbl
    USING
    (
        SELECT
            CAST(-1 AS BIGINT) AS product_sk,
            '-1' AS product_id,
            '-1' AS sku,
            'UNKNOWN' AS product_name,
            'UNKNOWN' AS brand,
            CAST(0 AS DECIMAL(18,4)) AS unit_price,
            'UNKNOWN' AS currency_code,
            FALSE AS is_active,
            'SYSTEM' AS attribute_hash,
            TIMESTAMP '1900-01-01 00:00:00' AS effective_start_date,
            TIMESTAMP '9999-12-31 00:00:00' AS effective_end_date,
            TRUE AS is_current,
            0 AS row_version,
            'SYSTEM' AS _source_batch_id,
        'SYSTEM' AS _pipeline_run_id,
            current_timestamp() AS _gold_loaded_timestamp
    ) AS src_tbl

    ON tgt_tbl.product_sk = src_tbl.product_sk
    WHEN NOT MATCHED
    THEN INSERT *;

ALTER TABLE ${gold}.dim_product ADD CONSTRAINT ck_dim_product_effective_range CHECK (effective_start_date < effective_end_date);
ALTER TABLE ${gold}.dim_product ADD CONSTRAINT ck_dim_product_price CHECK (unit_price >= 0);
ALTER TABLE ${gold}.dim_product ADD CONSTRAINT ck_dim_product_row_version CHECK (row_version >= 0)


In [ ]:
%%sql

-- ============================================================
-- DDL: fact_payment
-- Gold Layer — Payment Transaction Fact Table
-- ============================================================
-- Grain:            One row per payment
-- Source:           silver_payment
-- Dimension FKs:    customer_sk, payment_date_sk
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.fact_payment
(
    -- Surrogate key
    payment_sk BIGINT NOT NULL,

    -- Degenerate dimensions
    payment_id STRING NOT NULL,
    order_id STRING NOT NULL,

    -- Foreign keys to dimensions
    customer_sk BIGINT NOT NULL,
    payment_date_sk BIGINT NOT NULL,

    -- Payment attributes
    payment_method STRING,
    payment_status STRING,
    currency_code STRING,
    transaction_reference STRING,

    -- Measures
    payment_amount DECIMAL(18,2),

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints (informational — used by the optimiser)
    CONSTRAINT fact_payment_pk PRIMARY KEY (payment_sk),

    -- Foreign keys to dimension tables
    CONSTRAINT fk_fact_payment_customer FOREIGN KEY (customer_sk) REFERENCES ${gold}.dim_customer(customer_sk),
    CONSTRAINT fk_fact_payment_date FOREIGN KEY (payment_date_sk) REFERENCES ${gold}.dim_date(date_sk)
)
USING DELTA;

ALTER TABLE ${gold}.fact_payment ADD CONSTRAINT fact_payment_bk UNIQUE (payment_id);
ALTER TABLE ${gold}.fact_payment ADD CONSTRAINT chk_payment_amount CHECK (payment_amount >= 0);
ALTER TABLE ${gold}.fact_payment ADD CONSTRAINT chk_payment_status CHECK (payment_status IN ('PENDING', 'CAPTURED', 'FAILED', 'REFUNDED'));
ALTER TABLE ${gold}.fact_payment ADD CONSTRAINT chk_currency_code CHECK (LENGTH(currency_code) = 3);


In [ ]:
%%sql

-- ============================================================
-- DDL: fact_return
-- Gold Layer — Return Transaction Fact Table
-- ============================================================
-- Grain:            One row per return item
-- Source:           silver_return + silver_return_item
-- Dimension FKs:    customer_sk, product_sk, return_date_sk
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.fact_return
(
    -- Surrogate key
    return_sk BIGINT NOT NULL,

    -- Degenerate dimensions
    return_id STRING NOT NULL,
    return_item_id STRING NOT NULL,
    order_item_id STRING NOT NULL,

    -- Foreign keys to dimensions
    customer_sk BIGINT NOT NULL,
    product_sk BIGINT NOT NULL,
    return_date_sk BIGINT NOT NULL,

    -- Return attributes
    return_reason STRING,
    return_status STRING,

    -- Measures
    quantity INT,
    refund_amount DECIMAL(18,2),

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints (informational — used by the optimiser)
    CONSTRAINT fact_return_pk PRIMARY KEY (return_sk),

    -- Foreign keys to dimension tables
    CONSTRAINT fk_fact_return_customer FOREIGN KEY (customer_sk) REFERENCES ${gold}.dim_customer(customer_sk),
    CONSTRAINT fk_fact_return_product FOREIGN KEY (product_sk) REFERENCES ${gold}.dim_product(product_sk),
    CONSTRAINT fk_fact_return_date FOREIGN KEY (return_date_sk) REFERENCES ${gold}.dim_date(date_sk)
)
USING DELTA;

ALTER TABLE ${gold}.fact_return ADD CONSTRAINT fact_return_bk UNIQUE (return_item_id);
ALTER TABLE ${gold}.fact_return ADD CONSTRAINT chk_return_quantity CHECK (quantity > 0);
ALTER TABLE ${gold}.fact_return ADD CONSTRAINT chk_refund_amount CHECK (refund_amount >= 0);
ALTER TABLE ${gold}.fact_return ADD CONSTRAINT chk_return_status CHECK (return_status IN ('PENDING', 'APPROVED', 'REJECTED', 'COMPLETED'));
ALTER TABLE ${gold}.fact_return ADD CONSTRAINT chk_return_reason CHECK (return_reason IS NOT NULL)

In [ ]:
%%sql

-- ============================================================
-- DDL: fact_sales
-- Gold Layer — Sales Transaction Fact Table
-- ============================================================
-- Grain:            One row per order item
-- Source:           silver_order + silver_order_item
-- Dimension FKs:    customer_sk, billing_address_sk,
--                   shipping_address_sk, product_sk,
--                   order_date_sk
-- ============================================================

CREATE OR REPLACE TABLE ${gold}.fact_sales
(
    -- Surrogate key
    sales_sk BIGINT NOT NULL,

    -- Degenerate dimensions
    order_id STRING NOT NULL,
    order_item_id STRING NOT NULL,

    -- Foreign keys to dimensions
    customer_sk BIGINT NOT NULL,
    billing_address_sk BIGINT NOT NULL,
    shipping_address_sk BIGINT NOT NULL,
    product_sk BIGINT NOT NULL,
    order_date_sk BIGINT NOT NULL,

    -- Order context (from silver_order)
    channel STRING,
    order_status STRING,
    currency_code STRING,

    -- Measures (from silver_order_item)
    quantity INT,
    unit_price DECIMAL(18,2),
    discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2),
    line_total_amount DECIMAL(18,2),

    -- Audit / lineage
    _source_batch_id STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL,
    _gold_loaded_timestamp TIMESTAMP NOT NULL,

    -- Constraints (informational — used by the optimiser)
    CONSTRAINT fact_sales_pk PRIMARY KEY (sales_sk),

    -- Foreign keys to dimension tables
    CONSTRAINT fk_fact_sales_customer FOREIGN KEY (customer_sk) REFERENCES ${gold}.dim_customer(customer_sk),
    CONSTRAINT fk_fact_sales_billing_addr FOREIGN KEY (billing_address_sk) REFERENCES ${gold}.dim_address(address_sk),
    CONSTRAINT fk_fact_sales_shipping_addr FOREIGN KEY (shipping_address_sk) REFERENCES ${gold}.dim_address(address_sk),
    CONSTRAINT fk_fact_sales_product FOREIGN KEY (product_sk) REFERENCES ${gold}.dim_product(product_sk),
    CONSTRAINT fk_fact_sales_date FOREIGN KEY (order_date_sk) REFERENCES ${gold}.dim_date(date_sk)
)
USING DELTA;

ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT fact_sales_bk UNIQUE (order_item_id);
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_sales_quantity CHECK (quantity > 0);
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_unit_price CHECK (unit_price >= 0);
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_discount_amount CHECK (discount_amount >= 0);
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_tax_amount CHECK (tax_amount >= 0);
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_line_total CHECK (line_total_amount >= 0);
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_order_status CHECK (order_status IN ('PENDING', 'COMPLETED', 'CANCELLED'));
ALTER TABLE ${gold}.fact_sales ADD CONSTRAINT chk_currency_code CHECK (LENGTH(currency_code) = 3);
